<a href="https://colab.research.google.com/github/maneesha-bogahawatta/J26-DS-320/blob/member4%2Fcritical-incidents/components/c4_critical_incidents/fall_branch/C4_fall_detection_combined_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, copy, pickle
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# GMDCSA arrays saved earlier
d = pickle.load(open('/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_arrays.pkl', 'rb'))
X, y, subj = d['X'], d['y'], d['subj']
print(X.shape, y.shape, sorted(set(subj)))

In [ ]:
import pickle
from sklearn.model_selection import train_test_split

with open('/content/drive/MyDrive/J26-DS-320/datasets/urfd_features.pkl', 'rb') as f:
    urfd_seqs = pickle.load(f)

FEATURE_NAMES = ["nose_y", "shoulder_y", "hip_y", "aspect_ratio", "width", "height"]
T = 100

def resample_by_index(feats, T=T):
    arr = np.array([[f[n] for n in FEATURE_NAMES] for f in feats], dtype=np.float64)
    grid = np.linspace(0, len(arr) - 1, T)
    return np.stack([np.interp(grid, np.arange(len(arr)), arr[:, j]) for j in range(arr.shape[1])], axis=1).astype(np.float32)

Xu = np.array([resample_by_index(s['features']) for s in urfd_seqs])
yu = np.array([s['label'] for s in urfd_seqs])
nu = [s['name'] for s in urfd_seqs]

# EXACT original split (same parameters as your notebook, same order of sequences)
idx = np.arange(len(yu))
i_tr, i_tmp, _, y_tmp = train_test_split(idx, yu, test_size=0.3, stratify=yu, random_state=42)
i_va, i_te = train_test_split(i_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

Xu_tr, yu_tr = Xu[i_tr], yu[i_tr]
Xu_va, yu_va = Xu[i_va], yu[i_va]
Xu_te, yu_te = Xu[i_te], yu[i_te]
print('URFD train/val/test:', len(i_tr), len(i_va), len(i_te))
print('test names:', [nu[i] for i in i_te])

In [ ]:
class FallCNNLSTM(nn.Module):
    def __init__(self, num_features=6, cnn_channels=[32, 64, 128], lstm_hidden=128, lstm_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, cnn_channels[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(cnn_channels[1], cnn_channels[2], kernel_size=3, padding=1)
        self.relu = nn.ReLU(); self.dropout = nn.Dropout(0.3)
        self.lstm = nn.LSTM(cnn_channels[2], lstm_hidden, num_layers=lstm_layers, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(lstm_hidden, 64); self.fc2 = nn.Linear(64, 2)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = self.dropout(self.relu(self.conv3(x)))
        _, (h_n, _) = self.lstm(x.permute(0, 2, 1))
        return self.fc2(self.relu(self.fc1(h_n[-1])))

def loader(Xa, ya, shuffle):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya, dtype=torch.long)), batch_size=8, shuffle=shuffle)

def predict(model, Xa):
    model.eval()
    with torch.no_grad():
        return torch.argmax(model(torch.tensor(Xa).to(device)), dim=1).cpu().numpy()

def train_model(Xtr, ytr, Xva, yva, epochs=50, patience=10):
    model = FallCNNLSTM().to(device)
    w = torch.tensor([1.0 / c for c in np.bincount(ytr)], dtype=torch.float32).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for bx, by in loader(Xtr, ytr, True):
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad(); crit(model(bx), by).backward(); opt.step()
        f1 = f1_score(yva, predict(model, Xva), zero_division=0)
        if f1 > best_f1: best_f1, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model

def metrics(yt, yp):
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    return {"precision": float(precision_score(yt, yp, zero_division=0)),
            "recall": float(recall_score(yt, yp, zero_division=0)),
            "f1": float(f1_score(yt, yp, zero_division=0)),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

In [ ]:
# ---------- C1: leave-one-GMDCSA-subject-out, URFD (train+val only) always in training ----------
torch.manual_seed(42); np.random.seed(42)
u_X = np.concatenate([Xu_tr, Xu_va]); u_y = np.concatenate([yu_tr, yu_va])

c1, t_all, p_all = {}, [], []
for held in sorted(set(subj)):
    te = subj == held
    gX, gy = X[~te], y[~te]
    gX_tr, gX_va, gy_tr, gy_va = train_test_split(gX, gy, test_size=0.15, stratify=gy, random_state=42)
    Xtr = np.concatenate([u_X, gX_tr]); ytr = np.concatenate([u_y, gy_tr])
    m = train_model(Xtr, ytr, gX_va, gy_va)
    yp = predict(m, X[te])
    c1[held] = metrics(y[te], yp)
    t_all += list(y[te]); p_all += list(yp)
    print(held, c1[held])
c1_pooled = metrics(np.array(t_all), np.array(p_all))
print('\nC1 POOLED (URFD + GMDCSA):', c1_pooled)
print('GMDCSA-only pooled was: f1=0.901  precision=0.837  recall=0.975')

# ---------- C2: final combined model, tested on the original URFD test set ----------
torch.manual_seed(42); np.random.seed(42)
gX_tr, gX_va, gy_tr, gy_va = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
Xtr = np.concatenate([Xu_tr, gX_tr]); ytr = np.concatenate([yu_tr, gy_tr])
Xva = np.concatenate([Xu_va, gX_va]); yva = np.concatenate([yu_va, gy_va])
final = train_model(Xtr, ytr, Xva, yva)
c2 = metrics(yu_te, predict(final, Xu_te))
print('\nC2 combined model on original URFD test (11 seq):', c2, '| original URFD-only model: f1=0.889')

torch.save(final.state_dict(), '/content/drive/MyDrive/J26-DS-320/datasets/fall_cnn_lstm_v2.pt')
json.dump({"c1_loso_with_urfd_per_subject": c1, "c1_pooled": c1_pooled,
           "gmdcsa_only_pooled_f1": 0.9006, "c2_urfd_test": c2, "urfd_only_f1": 0.889},
          open('/content/drive/MyDrive/J26-DS-320/datasets/combined_model_results.json', 'w'), indent=2)

In [ ]:
  from google.colab import files
  files.download('/content/drive/MyDrive/J26-DS-320/datasets/combined_model_results.json')

In [ ]:
def predict_proba(model, Xa):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(Xa).to(device)), dim=1)[:, 1].cpu().numpy()

USE_URFD = False   # False = GMDCSA-only (our best so far); True = add URFD to training
SEEDS = [0, 1, 2]

oof = np.zeros(len(y))
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    for held in sorted(set(subj)):
        te = subj == held
        gX_tr, gX_va, gy_tr, gy_va = train_test_split(X[~te], y[~te], test_size=0.15,
                                                       stratify=y[~te], random_state=seed)
        if USE_URFD:
            Xtr = np.concatenate([u_X, gX_tr]); ytr = np.concatenate([u_y, gy_tr])
        else:
            Xtr, ytr = gX_tr, gy_tr
        m = train_model(Xtr, ytr, gX_va, gy_va)
        oof[te] += predict_proba(m, X[te]) / len(SEEDS)
    print('seed', seed, 'done')

# ---- sweep the cutoff ----
grid = np.round(np.arange(0.30, 0.96, 0.05), 2)
print(f'\n{"thr":>5} {"prec":>6} {"recall":>7} {"f1":>6} {"FP":>4} {"FN":>4}')
rows = {}
for t in grid:
    mt = metrics(y, (oof >= t).astype(int)); rows[float(t)] = mt
    print(f'{t:5.2f} {mt["precision"]:6.3f} {mt["recall"]:7.3f} {mt["f1"]:6.3f} {mt["fp"]:4d} {mt["fn"]:4d}')

# ---- fair check: pick the cutoff on 3 subjects, apply it to the 4th ----
def best_thr(mask):
    return max(grid, key=lambda t: f1_score(y[mask], (oof[mask] >= t).astype(int), zero_division=0))

pred_cv, thr_used = np.zeros(len(y), dtype=int), {}
for held in sorted(set(subj)):
    te = subj == held
    thr_used[held] = float(best_thr(~te))
    pred_cv[te] = (oof[te] >= thr_used[held]).astype(int)
cv = metrics(y, pred_cv)
print('\nthresholds chosen per held-out subject:', thr_used)
print('FAIR (cross-subject) tuned result:', cv)
print('default 0.50 for reference:       ', metrics(y, (oof >= 0.5).astype(int)))

json.dump({"use_urfd": USE_URFD, "seeds": SEEDS, "sweep": rows,
           "thresholds_per_subject": thr_used, "cross_subject_tuned": cv},
          open('/content/drive/MyDrive/J26-DS-320/datasets/threshold_tuning_results.json', 'w'), indent=2)
np.save('/content/drive/MyDrive/J26-DS-320/datasets/loso_oof_probs.npy', oof)

In [ ]:
import pandas as pd

In [ ]:
names = d['names']                      # e.g. 'Subject 1/ADL/08.mp4'
THR = 0.40
pred = (oof >= THR).astype(int)

# look up each clip's description and labelled classes from the CSVs
ROOT = '/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24'
lookup = {}
for sj in sorted(set(subj)):
    for cat in ['ADL', 'Fall']:
        df = pd.read_csv(f'{ROOT}/{sj}/{cat}.csv'); df.columns = [c.strip() for c in df.columns]
        for _, r in df.iterrows():
            lookup[f'{sj}/{cat}/{r["File Name"]}'] = (str(r['Description']).strip(), str(r['Classes']).strip(), str(r['Attire']).strip())

def show(title, idxs):
    print(f'\n=== {title} ({len(idxs)}) ===')
    for i in sorted(idxs, key=lambda i: -oof[i] if y[i]==0 else oof[i]):
        desc, cls, attire = lookup.get(names[i], ('?', '?', '?'))
        print(f'{names[i]:26s} p={oof[i]:.2f} | {desc} | {cls} | {attire}')

show('FALSE ALARMS (normal clips flagged as fall)', [i for i in range(len(y)) if y[i]==0 and pred[i]==1])
show('MISSED FALLS',                                  [i for i in range(len(y)) if y[i]==1 and pred[i]==0])

# which subject/attire do the false alarms come from?
fa = [i for i in range(len(y)) if y[i]==0 and pred[i]==1]
print('\nfalse alarms per subject:', pd.Series([subj[i] for i in fa]).value_counts().to_dict())
print('false alarms per attire :', pd.Series([lookup[names[i]][2] for i in fa]).value_counts().to_dict())

In [ ]:
import pandas as pd
oof_old = oof.copy()    # baseline probabilities (6 features) for comparison

results = pickle.load(open('/content/drive/MyDrive/J26-DS-320/datasets/gmdcsa24_features.pkl', 'rb'))
names = d['names']
FEATURE_NAMES = ["nose_y", "shoulder_y", "hip_y", "aspect_ratio", "width", "height"]
VEL_IDX = [0, 1, 2, 5]      # nose_y, shoulder_y, hip_y, height -> velocity per second

def clip_to_array10(feats, T=100):
    valid = [f for f in feats if f is not None]
    if len(valid) < 5: return None
    t = np.array([f['t'] for f in valid])
    P = np.array([[f[n] for n in FEATURE_NAMES] for f in valid])
    k = np.ones(3) / 3                                   # light smoothing to tame pose jitter
    Ps = np.stack([np.convolve(np.pad(P[:, j], (1, 1), mode='edge'), k, mode='valid') for j in range(P.shape[1])], axis=1)
    V = np.clip(np.stack([np.gradient(Ps[:, j], t) for j in VEL_IDX], axis=1), -5, 5)
    A = np.concatenate([P, V], axis=1)                   # (frames, 10)
    grid = np.linspace(t.min(), t.max(), T)
    return np.stack([np.interp(grid, t, A[:, j]) for j in range(A.shape[1])], axis=1).astype(np.float32)

X10 = np.array([clip_to_array10(results[n]['features']) for n in names])
print(X10.shape)     # expect (160, 100, 10)

In [ ]:
def train_model(Xtr, ytr, Xva, yva, epochs=50, patience=10):     # now handles any number of features
    model = FallCNNLSTM(num_features=Xtr.shape[2]).to(device)
    w = torch.tensor([1.0 / c for c in np.bincount(ytr)], dtype=torch.float32).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for bx, by in loader(Xtr, ytr, True):
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad(); crit(model(bx), by).backward(); opt.step()
        f1 = f1_score(yva, predict(model, Xva), zero_division=0)
        if f1 > best_f1: best_f1, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model

oof_vel = np.zeros(len(y))
for seed in SEEDS:                       # same 3 seeds as before
    torch.manual_seed(seed); np.random.seed(seed)
    for held in sorted(set(subj)):
        te = subj == held
        Xtr, Xva, ytr, yva = train_test_split(X10[~te], y[~te], test_size=0.15, stratify=y[~te], random_state=seed)
        m = train_model(Xtr, ytr, Xva, yva)
        oof_vel[te] += predict_proba(m, X10[te]) / len(SEEDS)
    print('seed', seed, 'done')

print('\n            prec   recall   f1    FP  FN')
for label, p in [('6 features ', oof_old), ('10 features', oof_vel)]:
    for t in (0.40, 0.50):
        mt = metrics(y, (p >= t).astype(int))
        print(f'{label} @{t:.2f}  {mt["precision"]:.3f}  {mt["recall"]:.3f}  {mt["f1"]:.3f}  {mt["fp"]:3d} {mt["fn"]:3d}')

# did the three problem groups improve? (the 18 clips flagged before at 0.40)
old_fa = np.where((y == 0) & (oof_old >= 0.40))[0]
still = [i for i in old_fa if oof_vel[i] >= 0.40]
print(f'\nOf the {len(old_fa)} earlier false alarms, still flagged with velocity features: {len(still)}')
print('missed-fall clip probability:', {names[i]: round(float(oof_vel[i]), 2) for i in np.where(y == 1)[0] if oof_old[i] < 0.40})

json.dump({"six_features": {str(t): metrics(y, (oof_old >= t).astype(int)) for t in (0.4, 0.5)},
           "ten_features": {str(t): metrics(y, (oof_vel >= t).astype(int)) for t in (0.4, 0.5)},
           "earlier_false_alarms_still_flagged": len(still), "of": int(len(old_fa))},
          open('/content/drive/MyDrive/J26-DS-320/datasets/velocity_features_results.json', 'w'), indent=2)
np.save('/content/drive/MyDrive/J26-DS-320/datasets/loso_oof_probs_vel.npy', oof_vel)

In [ ]:
  from google.colab import files
  base = '/content/drive/MyDrive/J26-DS-320/datasets/'
  files.download(base + 'velocity_features_results.json')
  files.download(base + 'threshold_tuning_results.json')